In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG         = "clutchlytics"
BRONZE_GAMES    = f"{CATALOG}.bronze.raw_nhl_games"
BRONZE_SCOREBOARD = f"{CATALOG}.bronze.raw_nhl_scoreboard"
DIM_TEAMS       = f"{CATALOG}.silver.dimTeams"
SILVER_TABLE    = f"{CATALOG}.silver.dimGames"
 
SPORT  = "hockey"
LEAGUE = "nhl"
 
print(f"Primary source   : {BRONZE_GAMES}")
print(f"Enrichment source: {BRONZE_SCOREBOARD}")
print(f"dimTeams ref     : {DIM_TEAMS}")
print(f"Target           : {SILVER_TABLE}")
print(f"Sport            : {SPORT}")
print(f"League           : {LEAGUE}")

In [0]:
# ── READ SOURCES ──────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
 
games_df      = spark.table(BRONZE_GAMES)
scoreboard_df = spark.table(BRONZE_SCOREBOARD)
 
print(f"Bronze games rows      : {games_df.count()}")
print(f"Bronze scoreboard rows : {scoreboard_df.count()}")

In [0]:
# ── JOIN dimTeams — RESOLVE clutch_team_id ────────────────────────────────────────
# Join on team_id + league (not abbreviation) — ESPN team_id is stable.
# dimTeams.team_id is INT, bronze games team_id is STRING — cast to match.
 
dim_teams = (
    spark.table(DIM_TEAMS)
    .filter(F.col("league") == LEAGUE)
    .select(
        F.col("clutch_team_id"),
        F.col("team_id").cast("string").alias("dim_team_id"),
        F.col("abbreviation").alias("dim_abbreviation"),
    )
)
 
print(f"dimTeams rows for {LEAGUE}: {dim_teams.count()}")
 
# ── Enrich scoreboard — extract short_name and series_total_games ──
# One row per game in scoreboard — join to games on event_id
scoreboard_enrichment = (
    scoreboard_df
    .select(
        F.col("event_id").alias("sb_event_id"),
        F.col("short_name"),
        F.col("series_total_games"),
    )
    .dropDuplicates(["sb_event_id"])
)

In [0]:
# ── BUILD BASE GAMES TABLE ────────────────────────────────────────────────────
 
base_df = (
    games_df
    # ── Join scoreboard enrichment ──
    .join(
        scoreboard_enrichment,
        games_df.event_id == scoreboard_enrichment.sb_event_id,
        how="left"
    )
    # ── Join dimTeams for home team ──
    .join(
        dim_teams.select(
            F.col("clutch_team_id").alias("clutch_home_team_id"),
            F.col("dim_team_id").alias("home_dim_team_id"),
            F.col("dim_abbreviation").alias("home_abbr_check"),
        ),
        games_df.home_team_id == F.col("home_dim_team_id"),
        how="left"
    )
    # ── Join dimTeams for away team ──
    .join(
        dim_teams.select(
            F.col("clutch_team_id").alias("clutch_away_team_id"),
            F.col("dim_team_id").alias("away_dim_team_id"),
            F.col("dim_abbreviation").alias("away_abbr_check"),
        ),
        games_df.away_team_id == F.col("away_dim_team_id"),
        how="left"
    )
    .select(
        # ── Identity ──
        games_df.event_id.alias("source_event_id"),
        F.lit(None).cast("string").alias("fct_event_key"),  # NULL — placeholder for fctGames FK
        F.lit(SPORT).alias("sport"),
        F.lit(LEAGUE).alias("league"),
 
        # ── Teams ──
        F.col("clutch_home_team_id"),
        F.col("clutch_away_team_id"),
        games_df.home_team.alias("home_team_abbr"),
        games_df.away_team.alias("away_team_abbr"),
 
        # ── Schedule attributes ──
        F.to_date(games_df.game_date).alias("game_date"),
        games_df.game_name,
        scoreboard_enrichment.short_name,  # Use scoreboard short_name explicitly
        games_df.venue,  # from games_df
        games_df.season.cast("integer").alias("season"),
        games_df.season_type,
        F.when(games_df.season_type == "playoffs", True)
         .otherwise(False)
         .alias("is_playoff"),
 
        # ── Playoff context ──
        games_df.round.cast("integer").alias("round"),
        scoreboard_enrichment.series_total_games.cast("integer").alias("series_total_games"),  # from scoreboard explicitly
 
        # ── Metadata ──
        games_df.ingested_at.alias("bronze_ingested_at"),
    )
    .dropDuplicates(["source_event_id", "league"])
)
 
print(f"Base rows after joins: {base_df.count()}")
 
# ── Warn on unmatched team joins ──
unmatched_home = base_df.filter(F.col("clutch_home_team_id").isNull()).count()
unmatched_away = base_df.filter(F.col("clutch_away_team_id").isNull()).count()
print(f"Unmatched home teams : {unmatched_home}  {'✓' if unmatched_home == 0 else '<-- investigate'}")
print(f"Unmatched away teams : {unmatched_away}  {'✓' if unmatched_away == 0 else '<-- investigate'}")

In [0]:
# ── DERIVE series_key AND game_number_in_series ───────────────────────────────
# series_key: MIN(home_team_id, away_team_id)_MAX(home_team_id, away_team_id)_R{round}
# Consistent regardless of home/away — identifies a unique playoff series.
# game_number_in_series: rank by game_date within series_key (playoffs only).
 
enriched_df = (
    base_df
    # ── Build series_key using clutch team IDs for consistency ──
    .withColumn(
        "series_key",
        F.when(
            F.col("is_playoff") == True,
            F.concat(
                F.least(
                    F.col("clutch_home_team_id").cast("string"),
                    F.col("clutch_away_team_id").cast("string")
                ),
                F.lit("_"),
                F.greatest(
                    F.col("clutch_home_team_id").cast("string"),
                    F.col("clutch_away_team_id").cast("string")
                ),
                F.lit("_R"),
                F.col("round").cast("string")
            )
        ).otherwise(F.lit(None))
    )
)
 
# ── Rank by game_date within series_key for playoffs ──
playoff_window = Window.partitionBy("series_key").orderBy("game_date")
 
enriched_df = (
    enriched_df
    .withColumn(
        "game_number_in_series",
        F.when(
            F.col("is_playoff") == True,
            F.rank().over(playoff_window)
        ).otherwise(F.lit(None))
    )
)
 
print("Series key and game number derived.")
print("\nSample — playoff games with series context:")
enriched_df.filter(F.col("is_playoff") == True).select(
    "source_event_id", "home_team_abbr", "away_team_abbr",
    "game_date", "series_key", "game_number_in_series", "round"
).orderBy("series_key", "game_date").show(20, truncate=False)

In [0]:
# ── ASSIGN clutch_game_id ─────────────────────────────────────────────────────
# Surrogate PK — sequential, continues from current MAX if table exists.
# Ordered by game_date then source_event_id for consistency.
 
table_exists = spark.catalog.tableExists(SILVER_TABLE)
 
if table_exists:
    max_id = spark.sql(f"""
        SELECT COALESCE(MAX(clutch_game_id), 0) AS max_id
        FROM {SILVER_TABLE}
    """).collect()[0]["max_id"]
 
    existing_leagues = spark.sql(f"""
        SELECT DISTINCT league FROM {SILVER_TABLE}
    """).rdd.flatMap(lambda x: x).collect()
 
    print(f"Table exists. Max clutch_game_id : {max_id}")
    print(f"Existing leagues                 : {existing_leagues}")
    is_first_load = LEAGUE not in existing_leagues
 
else:
    max_id           = 0
    existing_leagues = []
    is_first_load    = True
    print("Table does not exist — first load.")
 
from pyspark.sql.window import Window as W
 
id_window = W.orderBy("game_date", "source_event_id")
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
silver_df = (
    enriched_df
    .withColumn(
        "clutch_game_id",
        F.row_number().over(id_window) + max_id
    )
    .withColumn("ingested_at", F.lit(ingested_at))
    # ── Final column order ──
    .select(
        "clutch_game_id",
        "source_event_id",
        "fct_event_key",
        "sport",
        "league",
        "clutch_home_team_id",
        "clutch_away_team_id",
        "home_team_abbr",
        "away_team_abbr",
        "game_date",
        "game_name",
        "short_name",
        "venue",
        "season",
        "season_type",
        "is_playoff",
        "round",
        "series_key",
        "game_number_in_series",
        "series_total_games",
        "ingested_at",
        F.lit("bronze.raw_nhl_games").alias("source_table"),
    )
)
 
print(f"\nSample — first 5 rows:")
silver_df.show(5, truncate=False)

In [0]:
# ── WRITE TO SILVER ───────────────────────────────────────────────────────────
# First load: create table.
# Same league re-run: MERGE on source_event_id + league.
# New league: APPEND continuing clutch_game_id sequence.
 
if not table_exists:
    (
        silver_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Table created: {SILVER_TABLE}")
 
elif LEAGUE in existing_leagues:
    silver_df.createOrReplaceTempView("new_games")
 
    spark.sql(f"""
        MERGE INTO {SILVER_TABLE} AS target
        USING new_games AS source
        ON  target.source_event_id = source.source_event_id
        AND target.league          = source.league
        WHEN MATCHED THEN UPDATE SET
            fct_event_key          = source.fct_event_key,
            clutch_home_team_id    = source.clutch_home_team_id,
            clutch_away_team_id    = source.clutch_away_team_id,
            home_team_abbr         = source.home_team_abbr,
            away_team_abbr         = source.away_team_abbr,
            game_date              = source.game_date,
            game_name              = source.game_name,
            short_name             = source.short_name,
            venue                  = source.venue,
            season_type            = source.season_type,
            is_playoff             = source.is_playoff,
            series_key             = source.series_key,
            game_number_in_series  = source.game_number_in_series,
            series_total_games     = source.series_total_games,
            ingested_at            = source.ingested_at
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged updates for existing league: {LEAGUE}")
 
else:
    (
        silver_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(SILVER_TABLE)
    )
    print(f"Appended new league: {LEAGUE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
print("── Full dimGames ──")
spark.sql(f"""
    SELECT
        clutch_game_id,
        source_event_id,
        fct_event_key,
        sport,
        league,
        home_team_abbr,
        away_team_abbr,
        game_date,
        season_type,
        is_playoff,
        round,
        series_key,
        game_number_in_series,
        series_total_games,
        clutch_home_team_id,
        clutch_away_team_id
    FROM {SILVER_TABLE}
    ORDER BY game_date, series_key
""").show(50, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_games,
        COUNT(DISTINCT league)                                      AS leagues,
        COUNT(DISTINCT clutch_game_id)                             AS unique_clutch_ids,
        COUNT(CASE WHEN is_playoff = true  THEN 1 END)             AS playoff_games,
        COUNT(CASE WHEN is_playoff = false THEN 1 END)             AS regular_games,
        COUNT(DISTINCT series_key)                                  AS unique_series,
        COUNT(CASE WHEN game_number_in_series IS NULL
                    AND is_playoff = true THEN 1 END)              AS missing_game_numbers,
        COUNT(CASE WHEN clutch_home_team_id IS NULL THEN 1 END)    AS unmatched_home,
        COUNT(CASE WHEN clutch_away_team_id IS NULL THEN 1 END)    AS unmatched_away,
        COUNT(CASE WHEN fct_event_key IS NULL THEN 1 END)          AS null_fct_keys,
        MIN(clutch_game_id)                                        AS min_id,
        MAX(clutch_game_id)                                        AS max_id,
        MIN(game_date)                                             AS earliest_game,
        MAX(game_date)                                             AS latest_game
    FROM {SILVER_TABLE}
    WHERE league = '{LEAGUE}'
""")
 
print(f"Sanity checks ({LEAGUE}):")
checks.show(truncate=False)